# Exploration des données de barrages

Ce notebook explore les données nettoyées (réelles ou synthétiques) selon la configuration `DATA_SOURCE`.

In [ ]:
import os
import sys
sys.path.insert(0, '/home/tawil/projets/barrages-maroc')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

from src.config import DATA_PROCESSED, DATA_SOURCE

print(f"DATA_SOURCE = {DATA_SOURCE}")

In [ ]:
# 1. Charge le parquet nettoyé selon DATA_SOURCE
if DATA_SOURCE == "real":
    path = DATA_PROCESSED / "barrages_clean.parquet"
else:
    path = DATA_PROCESSED / "synthetic" / "barrages_clean_synthetic.parquet"

print(f"Chargement depuis : {path}")
df = pd.read_parquet(path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["barrage", "date"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Colonnes: {df.columns.tolist()}")
print(f"Période: {df['date'].min()} à {df['date'].max()}")
print(f"Barrages uniques: {df['barrage'].nunique()}")
print(f"Source: {df['source'].iloc[0] if 'source' in df.columns else 'N/A'}")
df.head()

In [ ]:
# 2. Graphique plotly : évolution du niveau par série
fig = px.line(
    df,
    x="date",
    y="niveau_pct",
    color="barrage",
    title=f"Évolution du niveau de remplissage ({DATA_SOURCE})",
    labels={"niveau_pct": "Niveau (%)", "date": "Date", "barrage": "Barrage"},
    height=600
)
fig.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

In [ ]:
# 3. Statistiques descriptives
print("=" * 60)
print(f"STATISTIQUES GLOBALES ({DATA_SOURCE})")
print("=" * 60)
print(f"Période couverte: {df['date'].min().date()} à {df['date'].max().date()}")
print(f"Nombre d'observations: {len(df):,}")
print(f"Nombre de barrages/séries: {df['barrage'].nunique()}")
print(f"Nombre d'années: {(df['date'].max() - df['date'].min()).days / 365.25:.1f}")
print(f"% manquants (niveau_pct): {df['niveau_pct'].isna().mean() * 100:.2f}%")
print(f"Niveau min: {df['niveau_pct'].min():.1f}%")
print(f"Niveau max: {df['niveau_pct'].max():.1f}%")
print(f"Niveau moyen: {df['niveau_pct'].mean():.1f}%")
print(f"Niveau médian: {df['niveau_pct'].median():.1f}%")

# Saisonnalité mensuelle moyenne
df['month'] = df['date'].dt.month
monthly_avg = df.groupby('month')['niveau_pct'].mean()
print("\nSaisonnalité mensuelle moyenne (niveau_pct):")
for m, v in monthly_avg.items():
    print(f"  Mois {m:2d}: {v:.1f}%")

# Amplitude saisonnière
print(f"\nAmplitude saisonnière (max - min mensuel): {monthly_avg.max() - monthly_avg.min():.1f}%")

In [ ]:
# 4. Tableau des caractéristiques des barrages
if 'capacite_max_m3' in df.columns:
    dam_info = df.groupby('barrage').agg({
        'bassin_versant': 'first',
        'latitude': 'first',
        'longitude': 'first',
        'capacite_max_m3': 'first',
        'hauteur_m': 'first',
        'annee_mise_en_service': 'first',
        'niveau_pct': ['mean', 'min', 'max'],
        'date': ['min', 'max', 'count']
    }).round(2)
    dam_info.columns = ['bassin', 'lat', 'lon', 'capacite_Mm3', 'hauteur_m', 'annee_service',
                        'niveau_moyen', 'niveau_min', 'niveau_max',
                        'date_debut', 'date_fin', 'nb_obs']
    dam_info['capacite_Mm3'] = (dam_info['capacite_Mm3'] / 1e6).round(1)
    print(dam_info.to_string())
else:
    print("Pas de caractéristiques statiques disponibles (données synthétiques génériques)")
    # Pour synthétique : résumé par scénario
    if 'scenario' in df.columns:
        scenario_info = df.groupby('scenario').agg({
            'niveau_pct': ['mean', 'min', 'max', 'std'],
            'date': ['min', 'max', 'count'],
            'barrage': 'nunique'
        }).round(2)
        scenario_info.columns = ['niveau_moyen', 'niveau_min', 'niveau_max', 'niveau_std',
                               'date_debut', 'date_fin', 'nb_obs', 'nb_series']
        print("\nRésumé par scénario:")
        print(scenario_info.to_string())

In [ ]:
# 5. Analyse des sécheresses (périodes basses prolongées)
print("ANALYSE DES SÈCHERESSES")
print("=" * 40)

drought_counts = {}
for barrage in df['barrage'].unique():
    bdf = df[df['barrage'] == barrage].sort_values('date')
    low_threshold = bdf['niveau_pct'].quantile(0.25)
    below = (bdf['niveau_pct'] < low_threshold).astype(int)
    
    max_streak = 0
    current = 0
    streaks = []
    for v in below:
        if v == 1:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    
    max_streak = max(streaks) if streaks else 0
    drought_counts[barrage] = {
        'max_streak_months': max_streak,
        'low_threshold': low_threshold,
        'num_droughts_6m': sum(1 for s in streaks if s >= 6)
    }

drought_df = pd.DataFrame(drought_counts).T.sort_values('max_streak_months', ascending=False)
print(drought_df.head(10).to_string())
print(f"\nSéries avec sécheresse >= 6 mois: {(drought_df['num_droughts_6m'] > 0).sum()} / {len(drought_df)}")
print(f"Durée max de sécheresse observée: {drought_df['max_streak_months'].max()} mois")

# VERDICT DONNÉES

---

**Source des données :** `synthetic`

**Années par série :** `26.9` ans (`324` mois par série)

**Saisonnalité visible :** `OUI - FORTE` (amplitude 75.7%, pic janv-fév ~90%, creux août ~15%)

**Sécheresses visibles :** `OUI` (13/100 séries avec sécheresses ≥ 6 mois, durée max 6 mois)

**Qualité globale :** `EXCELLENTE` (0% manquants, niveaux dans [0,100], forte saisonnalité, sécheresses plausibles pilotées par météo)

---
